In [3]:


import os
import pandas as pd
from eo_tides.model import model_phases
from eo_tides.utils import clip_models
from eo_tides.utils import list_models

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cm as cm




In [4]:
# # Clip tidal models to Cape Cod bounding box
# available_models, supported_models = list_models(
#     directory="tide_models/"
# )
# # Define bounding Box
# bbox_capecod = [-70.451122, 41.172320, -69.578025, 42.198560]
# clip_models(input_directory = "tide_models/",
#             output_directory = "tide_models_clipped",
#             model = "DTU23",
#             bbox = bbox_capecod)

In [9]:
# Get tide height data for Marconi Beach
start = "2024-10-01"
end = "2025-03-31"
tide_df= model_phases(
        x=-69.963189,
        y=41.896050, 
        time=pd.date_range(start, end, freq="15min"),
        model=["EOT20", "GOT5.5","GOT5.6","FES2022_extrapolated"],
        output_format = "wide",
        directory="tide_models_clipped/",
        return_tides = True,
)

tide_df["time"] = pd.date_range(start, end, freq="15min")
tide_df = tide_df.rename(columns={'Unnamed: 8': 'time'})
tide_df['time'] = pd.to_datetime(tide_df['time'])
col = tide_df.pop("time")
tide_df.insert(0, "time", col)

tide_df.to_csv(f"tidal_data_marconi_modelled_{start}-{end}.csv", index=False)

Modelling tides with EOT20, GOT5.5, GOT5.6, FES2022_extrapolated in parallel (models: 4, splits: 1)


100%|██████████| 4/4 [00:04<00:00,  1.04s/it]


Modelling tides with EOT20, GOT5.5, GOT5.6, FES2022_extrapolated in parallel (models: 4, splits: 1)


100%|██████████| 4/4 [00:03<00:00,  1.01it/s]


Converting to a wide format dataframe


In [ ]:

tide_columns= ["FES2022_extrapolated","EOT20","GOT5.5","GOT5.6"]
# Plotted FES2022b on each for comparison, couldn't find a better way to iterate the labels
labels = ["FES2022b","FES2022b","EOT20","FES2022b","GOT5.5","FES2022b","GOT5.6","FES2022b"]

cmap = plt.get_cmap('Spectral')
axes = df.plot(figsize = (10, 15),
                    x = "time",
                    y = tide_columns,
                    title="Modelled Tides at Marconi Beach, MA",
                    ylabel="Modelled Tide Height (m)",
                    xlabel="Date",
                    subplots=True,
                    grid=True,
                    x_compat=True,
                    legend=False,
                    cmap = cmap,
                    linewidth = 1.2);

for i,ax in enumerate(axes):
    ax.plot(df["time"], 
            df["FES2022_extrapolated"],
            color = "black",
            linewidth=4,
            alpha = 0.2,
            zorder = 3,
           )
    start_idx = i * 2
    end_idx = start_idx + 2
    
    # Pass the list of strings to the legend
    ax.legend(labels[start_idx:end_idx],loc='upper right')

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()
